# STRING Topology-Based Gene Ranking

This notebook builds a protein interaction graph from `../data/string_interactions_short.tsv`, computes four topological measures for each gene, creates binary labels from the median rule you requested, then trains the same three machine learning models used in the research paper:
- LASSO-style logistic regression
- SVM-RFE
- Random Forest

The final ranking is based on the average of the model scores after z-score normalization.

## Workflow assumptions

- the STRING interaction file is treated as an **undirected, unweighted** graph built from the gene-symbol columns `#node1` and `node2`
- duplicate edges and self-loops are removed
- labels are assigned as `1` only when a gene has values **greater than or equal to the median** for **all four** measures: betweenness, closeness, degree, and MCC
- for model training, all four features are standardized using training-set statistics only; the raw topology values are still preserved in the exported feature table for interpretation
- because the paper does not specify how many features SVM-RFE should retain, this notebook keeps the top 2 features

In [1]:
from __future__ import annotations

import math
import warnings
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings(
    'ignore',
    message="'penalty' was deprecated in version 1.8 and will be removed in 1.10.*",
)

DATA_PATH = Path('../data/string_interactions_short.tsv')
FEATURES_OUTPUT_PATH = Path('../data/string_topology_features.csv')
RANKING_OUTPUT_PATH = Path('../data/string_ml_gene_ranking.csv')

EDGE_SOURCE = '#node1'
EDGE_TARGET = 'node2'
NETWORK_FEATURES = ['betweenness', 'closeness', 'degree', 'mcc']
FEATURES_TO_STANDARDIZE = ['betweenness', 'closeness', 'degree', 'mcc']
RANDOM_STATE = 42

pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

In [2]:
edges = pd.read_csv(DATA_PATH, sep='\t')
required_edge_columns = [EDGE_SOURCE, EDGE_TARGET]
missing_edge_columns = [col for col in required_edge_columns if col not in edges.columns]
if missing_edge_columns:
    raise ValueError(f'Missing required edge columns: {missing_edge_columns}')

edges = edges[required_edge_columns].copy()
edges[EDGE_SOURCE] = edges[EDGE_SOURCE].astype(str).str.strip()
edges[EDGE_TARGET] = edges[EDGE_TARGET].astype(str).str.strip()
edges = edges[(edges[EDGE_SOURCE] != '') & (edges[EDGE_TARGET] != '')]
edges = edges[edges[EDGE_SOURCE] != edges[EDGE_TARGET]]

# Canonicalize undirected edges so A-B and B-A are treated as the same interaction.
edge_pairs = edges.apply(lambda row: tuple(sorted((row[EDGE_SOURCE], row[EDGE_TARGET]))), axis=1)
edges[['node_a', 'node_b']] = pd.DataFrame(edge_pairs.tolist(), index=edges.index)
edges = edges[['node_a', 'node_b']].drop_duplicates().reset_index(drop=True)

G = nx.from_pandas_edgelist(edges, source='node_a', target='node_b')

graph_summary = pd.Series(
    {
        'nodes': G.number_of_nodes(),
        'edges': G.number_of_edges(),
        'connected_components': nx.number_connected_components(G),
        'density': nx.density(G),
    }
)
graph_summary

nodes                    414.000000
edges                  8,841.000000
connected_components       1.000000
density                    0.103414
dtype: float64

In [3]:
degree_dict = dict(G.degree())
betweenness_dict = nx.betweenness_centrality(G, normalized=True)
closeness_dict = nx.closeness_centrality(G)


def compute_mcc(graph: nx.Graph) -> dict[str, int]:
    mcc = {node: 0 for node in graph.nodes}
    for clique in nx.find_cliques(graph):
        clique_weight = math.factorial(len(clique) - 1)
        for node in clique:
            mcc[node] += clique_weight
    return mcc


mcc_dict = compute_mcc(G)

features_df = pd.DataFrame(
    {
        'name': sorted(G.nodes()),
    }
)
features_df['betweenness'] = features_df['name'].map(betweenness_dict)
features_df['closeness'] = features_df['name'].map(closeness_dict)
features_df['degree'] = features_df['name'].map(degree_dict)
features_df['mcc'] = features_df['name'].map(mcc_dict)
features_df.head()

,name,betweenness,closeness,degree,mcc
0,ABL1,0.004266,0.544855,93,2360106612781915711989918
1,ACACA,0.001703,0.471461,28,19331905
2,ACACB,0.000267,0.403715,14,404090
3,ACE,0.011568,0.555855,95,747263680868044584
4,ACHE,0.003418,0.504274,35,47086838


In [4]:
medians = features_df[NETWORK_FEATURES].median()
medians

betweenness               0.000821
closeness                 0.479397
degree                   31.500000
mcc           1,060,863,764.000000
dtype: object

In [5]:
features_df['label'] = (
    (features_df['betweenness'] >= medians['betweenness'])
    & (features_df['closeness'] >= medians['closeness'])
    & (features_df['degree'] >= medians['degree'])
    & (features_df['mcc'] >= medians['mcc'])
).astype(int)

label_summary = pd.Series(
    {
        'positive_labels': int(features_df['label'].sum()),
        'negative_labels': int((features_df['label'] == 0).sum()),
        'positive_fraction': float(features_df['label'].mean()),
    }
)
label_summary

positive_labels     140.000000
negative_labels     274.000000
positive_fraction     0.338164
dtype: float64

In [6]:
features_df = features_df.sort_values('name').reset_index(drop=True)
features_df.to_csv(FEATURES_OUTPUT_PATH, index=False)
print(f'Topology features saved to: {FEATURES_OUTPUT_PATH.resolve()}')
features_df.head(10)

Topology features saved to: /Users/aman/Desktop/HubGenes/data/string_topology_features.csv


,name,betweenness,closeness,degree,mcc,label
0,ABL1,0.004266,0.544855,93,2360106612781915711989918,1
1,ACACA,0.001703,0.471461,28,19331905,0
2,ACACB,0.000267,0.403715,14,404090,0
3,ACE,0.011568,0.555855,95,747263680868044584,1
4,ACHE,0.003418,0.504274,35,47086838,0
5,ACVR1B,0.000016,0.377169,6,9,0
6,ADA,0.002759,0.468254,22,1826,0
7,ADAM10,0.001631,0.494019,39,4869794932,1
8,ADAM17,0.001046,0.500000,43,564757903595520,1
9,ADK,0.000852,0.368421,14,202,0


In [7]:
X = features_df[NETWORK_FEATURES].copy()
y = features_df['label'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

scaler = StandardScaler()
X_train = X_train.copy()
X_test = X_test.copy()
X_train[FEATURES_TO_STANDARDIZE] = scaler.fit_transform(X_train[FEATURES_TO_STANDARDIZE])
X_test[FEATURES_TO_STANDARDIZE] = scaler.transform(X_test[FEATURES_TO_STANDARDIZE])

split_summary = pd.Series(
    {
        'train_samples': len(X_train),
        'test_samples': len(X_test),
        'train_positive_labels': int(y_train.sum()),
        'test_positive_labels': int(y_test.sum()),
    }
)
split_summary

train_samples            331
test_samples              83
train_positive_labels    112
test_positive_labels      28
dtype: int64

In [8]:
X_all = features_df[NETWORK_FEATURES].copy()
X_all[FEATURES_TO_STANDARDIZE] = scaler.transform(features_df[FEATURES_TO_STANDARDIZE])

lasso_lr = LogisticRegression(
    penalty='elasticnet',
    l1_ratio=1.0,
    solver='saga',
    max_iter=10_000,
    random_state=RANDOM_STATE,
)
lasso_lr.fit(X_train.values, y_train.values)
score_lasso = lasso_lr.decision_function(X_all.values)

n_features_to_select = min(2, X_train.shape[1])
svm_rfe = RFE(
    estimator=LinearSVC(random_state=RANDOM_STATE, dual='auto'),
    n_features_to_select=n_features_to_select,
    step=1,
)
svm_rfe.fit(X_train.values, y_train.values)
score_svm_rfe = svm_rfe.estimator_.decision_function(svm_rfe.transform(X_all.values))

rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf.fit(X_train.values, y_train.values)
score_rf = rf.predict_proba(X_all.values)[:, 1]

selected_features = [feature for feature, keep in zip(NETWORK_FEATURES, svm_rfe.support_) if keep]
selected_features

['closeness', 'degree']

In [9]:
def zscore_1d(values: np.ndarray) -> np.ndarray:
    mean = float(np.mean(values))
    std = float(np.std(values))
    if std == 0:
        return np.zeros_like(values, dtype=float)
    return (values - mean) / std


z_lasso = zscore_1d(score_lasso)
z_svm_rfe = zscore_1d(score_svm_rfe)
z_rf = zscore_1d(score_rf)
composite_score = (z_lasso + z_svm_rfe + z_rf) / 3.0

ranking = pd.DataFrame(
    {
        'name': features_df['name'],
        'betweenness': features_df['betweenness'],
        'closeness': features_df['closeness'],
        'degree': features_df['degree'],
        'mcc': features_df['mcc'],
        'label': features_df['label'],
        'score_lasso': score_lasso,
        'score_svm_rfe': score_svm_rfe,
        'score_rf': score_rf,
        'z_lasso': z_lasso,
        'z_svm_rfe': z_svm_rfe,
        'z_rf': z_rf,
        'composite_score': composite_score,
    }
).sort_values('composite_score', ascending=False).reset_index(drop=True)

ranking['rank'] = np.arange(1, len(ranking) + 1)
ranking = ranking[
    [
        'rank', 'name', 'betweenness', 'closeness', 'degree', 'mcc', 'label',
        'score_lasso', 'score_svm_rfe', 'score_rf',
        'z_lasso', 'z_svm_rfe', 'z_rf', 'composite_score',
    ]
]

ranking.head(20)

,rank,name,betweenness,closeness,degree,mcc,label,score_lasso,score_svm_rfe,score_rf,z_lasso,z_svm_rfe,z_rf,composite_score
0,1,AKT1,0.057781,0.689482,229,52856376638766652931046169540,1,34.587268,9.436411,1.000000,4.906036,4.183831,1.439884,3.509917
1,2,TNF,0.057885,0.682645,223,52856411729207521029870864772,1,33.716471,9.110298,1.000000,4.787936,4.048051,1.439884,3.425290
2,3,SRC,0.054362,0.649371,195,52821201690573961969082469870,1,28.895163,7.549091,1.000000,4.134056,3.398026,1.439884,2.990655
3,4,ALB,0.039229,0.641304,185,52855247018991716231348539512,1,25.265256,7.101642,1.000000,3.641757,3.211726,1.439884,2.764456
4,5,EGFR,0.029766,0.641304,185,27390366370300340218577474062,1,23.746095,7.101642,1.000000,3.435724,3.211726,1.439884,2.695778
5,6,CASP3,0.017403,0.631498,174,52856491917775330605753152412,1,20.362751,6.582543,1.000000,2.976865,2.995594,1.439884,2.470781
6,7,BCL2,0.018180,0.629573,174,52856491917578355142232879346,1,20.326446,6.526991,1.000000,2.971941,2.972464,1.439884,2.461430
7,8,ESR1,0.032419,0.616418,161,52853223286835051703524620742,1,20.828171,5.868311,1.000000,3.039987,2.698216,1.439884,2.392696
8,9,HSP90AA1,0.018783,0.619190,163,52854533866032141247915491470,1,18.976327,5.991247,1.000000,2.788834,2.749402,1.439884,2.326040
9,10,NFKB1,0.010971,0.615499,159,52856486516579439192867160440,1,17.203211,5.798866,1.000000,2.548359,2.669302,1.439884,2.219182


In [10]:
evaluation = pd.DataFrame(
    [
        {
            'model': 'LASSO logistic regression',
            'test_accuracy': accuracy_score(y_test, lasso_lr.predict(X_test.values)),
            'test_roc_auc': roc_auc_score(y_test, lasso_lr.decision_function(X_test.values)),
        },
        {
            'model': 'SVM-RFE (LinearSVC)',
            'test_accuracy': accuracy_score(y_test, svm_rfe.predict(X_test.values)),
            'test_roc_auc': roc_auc_score(y_test, svm_rfe.estimator_.decision_function(svm_rfe.transform(X_test.values))),
        },
        {
            'model': 'Random Forest',
            'test_accuracy': accuracy_score(y_test, rf.predict(X_test.values)),
            'test_roc_auc': roc_auc_score(y_test, rf.predict_proba(X_test.values)[:, 1]),
        },
    ]
).sort_values('test_roc_auc', ascending=False)

evaluation

,model,test_accuracy,test_roc_auc
2,Random Forest,0.951807,0.996104
1,SVM-RFE (LinearSVC),0.927711,0.989610
0,LASSO logistic regression,0.927711,0.987662


In [11]:
ranking.to_csv(RANKING_OUTPUT_PATH, index=False)

print(f'Ranking saved to: {RANKING_OUTPUT_PATH.resolve()}')
print('Top 10 ranked genes:')
ranking[['rank', 'name', 'composite_score', 'label']].head(10)

Ranking saved to: /Users/aman/Desktop/HubGenes/data/string_ml_gene_ranking.csv
Top 10 ranked genes:


,rank,name,composite_score,label
0,1,AKT1,3.509917,1
1,2,TNF,3.425290,1
2,3,SRC,2.990655,1
3,4,ALB,2.764456,1
4,5,EGFR,2.695778,1
5,6,CASP3,2.470781,1
6,7,BCL2,2.461430,1
7,8,ESR1,2.392696,1
8,9,HSP90AA1,2.326040,1
9,10,NFKB1,2.219182,1


## Notes

- `string_topology_features.csv` stores the computed betweenness, closeness, degree, MCC, and median-based labels for all genes.
- `string_ml_gene_ranking.csv` stores the final ML-based ranking.
- if you want to adjust STRING confidence thresholds later, filter the input edge table before graph construction and rerun the notebook.